### SCENARIO 1

**TICKET**: "Our internal wiki at `wiki.corp.local` was working fine yesterday. This morning, nobody in the office can load it. Error in the browser is a generic 'can't reach this site.' IT hasn't changed anything on the wiki server as far as we know."

**SYSTEM**: Ubuntu 22.04 LTS / WSL2

**START POINT**: You're logged in as `dennie`, sudo available, on your own laptop which is on the same office network as the affected users.

#### SOLUTION — Step by step

**Step 1 — `ping` the hostname, with `-c` (never bare `ping`):**
```bash
$ ping -c 4 wiki.corp.local
ping: wiki.corp.local: Name or service not known
```
**Read it:** this error fires *before* any packet is sent — it's a **name-resolution failure**, not a network-reachability result. `ping hostname` secretly does two steps (resolve → send ICMP); this died at step 1. Don't confuse it with `100% packet loss`, which would mean resolution worked but nothing answered — a completely different fault, further along the stack.

**Step 2 — isolate the DNS failure with `dig`:**
```bash
$ dig wiki.corp.local +short
;; communications error to 10.50.0.5#53: timed out
;; communications error to 10.50.0.5#53: timed out
;; communications error to 10.50.0.5#53: timed out
;; no servers could be reached
```
**Read it:** not `NXDOMAIN` (record doesn't exist) and not `SERVFAIL` (server answered but errored) — this is a **transport-level failure reaching the resolver itself** (`10.50.0.5`, pulled straight from the error line). Nothing came back at all.

**Step 3 — separate "service down" from "whole host down" by pinging the resolver directly:**
```bash
$ ping -c 4 10.50.0.5
64 bytes from 10.50.0.5: icmp_seq=1 ttl=64 time=0.412 ms
...
4 packets transmitted, 4 received, 0% packet loss
```
**Read it:** the DNS server's **host** is alive and fast (sub-ms = same local subnet). So the machine itself didn't go down — but its **DNS service specifically** isn't answering queries. That's a service-level fault sitting on a healthy host.

**Step 4 — conclusion and escalation:**
You don't yet have a port-level check (`nc -zv 10.50.0.5 53`, or `ss -tlnp` run on that server — both come later in this pillar) to directly confirm "port 53 is closed/blocked," so the correct move with the current toolkit is to **escalate with precise evidence** rather than guess further:

> "`wiki.corp.local` fails to resolve for the whole office. The configured DNS server `10.50.0.5` responds to ping (host is up) but does not answer DNS queries at all (`dig` times out — `communications error ... no servers could be reached`). Looks like the DNS service on that box crashed, hung, or is being blocked on port 53. Needs to be checked/restarted by whoever owns that server."

**Extra points worth keeping in mind:**
- **Don't trust "nothing changed" at face value.** A crashed service, failed auto-update, full disk, or expired cert can all take a service down with zero deliberate human action. "Nothing changed" only means "no *deliberate* change" — not "nothing happened."
- **"Above/below the stack" framing:** DNS resolution **isn't a layer** in the ping-diagnosis model at all — it's a **separate pre-flight lookup step** that happens *before* the network-layer test even begins. Application is the top of the real stack; nothing sits "above" it.
- **Full error-message meanings** for everything seen here (and more) are in [9error-reference.ipynb](9error-reference.ipynb) — e.g. the 3-bucket triage: resolution failed vs transport failed vs reached-but-rejected.

### SCENARIO 2

**TICKET**: "Customers in our Berlin office say the checkout page on `shop.example.com` is **painfully slow** — takes 10-15 seconds to load, but eventually works. No errors shown. Other offices report no issues at all with the same site."

**SYSTEM**: Ubuntu 22.04 LTS / WSL2

**START POINT**: You're logged into a laptop on the Berlin office network, `dennie`, sudo available.

I think about **checking the route** for some router which could be lagging along the way. The packets travel clear but the speed is the main concern so it could reveal the router that is slow.

#### SOLUTION — Step by step

**Step 1 — baseline with `ping`:**
```shell
$ ping -c 4 shop.example.com
64 bytes from 203.0.113.44: icmp_seq=1 ttl=51 time=287 ms
...
4 packets transmitted, 4 received, 0% packet loss
rtt min/avg/max/mdev = 283.112/289.034/295.221/4.521 ms
```
**Read it:** `0% packet loss` — this is **not** a connectivity failure, nothing is broken in the binary up/down sense. But `~290ms` average RTT is high for what should likely be a much shorter hop if the target is reasonably close — that's your "slow, not down" signal, and the reason to reach for `traceroute` next instead of stopping here.

**Step 2 — find where the delay is introduced with `traceroute`:**
```shell
$ traceroute -n shop.example.com
 1  10.20.0.1       0.612 ms   0.589 ms   0.601 ms
 2  10.20.255.1     1.203 ms   1.150 ms   1.187 ms
 3  198.51.100.9  284.441 ms 286.102 ms 285.339 ms
 4  198.51.100.33 287.209 ms 288.014 ms 286.998 ms
 5  192.0.2.17    289.455 ms 290.112 ms 288.877 ms
 6  192.0.2.201   291.003 ms 289.774 ms 290.556 ms
 7  203.0.113.5   290.887 ms 291.204 ms 290.333 ms
 8  203.0.113.44  289.991 ms 290.447 ms 290.120 ms
```
**Read it, hop by hop:**
- Hops 1-2 (`10.20.x.x`, private IPs): sub-2ms — this is the Berlin office's own internal LAN, fast as expected.
- **Hop 3 (`198.51.100.9`) is where the jump happens** — from ~1ms to ~285ms in a single hop. This is the first hop *outside* the office network — i.e., the handoff to the ISP/uplink.
- Hops 4-8: latency stays **flat** around 285-291ms, not climbing further. A flat latency after the jump (rather than compounding increases) means one slow link is responsible, not cumulative congestion across many hops.
- Hop 8 = `203.0.113.44`, matching the IP `ping` resolved — **the trace reaches the destination successfully.** That clears the destination server itself of any blame.

**Step 3 — fold in the ticket's own context, not just the command output:**
The ticket already told you **other offices have zero issues with the same site.** If `shop.example.com`'s infrastructure were the problem, every office would be slow, not just Berlin. Combined with the hop-3 jump happening immediately after leaving the Berlin office's own gateway, this pins the fault specifically on **Berlin's local ISP/uplink** — not a generic "ISP or destination, take your pick."

**Step 4 — conclusion and escalation:**
> "Checkout page is slow (~290ms extra latency) only for the Berlin office; other offices are unaffected, and the destination (`shop.example.com`, `203.0.113.44`) is reached successfully in the trace. The delay is introduced at the very first hop leaving the Berlin office network (`198.51.100.9`) and stays flat afterward — points to Berlin's own ISP/uplink, not the destination. Escalate to the network/ISP team responsible for Berlin's internet circuit, not the app or destination's hosting team."

**Extra points worth keeping in mind:**
- **A ticket's framing is diagnostic evidence too** — "only this office," "only since yesterday," "only for this user" — fold these details into your escalation target, don't rely on command output in isolation.
- **Flat-after-a-jump vs climbing latency** are different stories: flat = one slow link; steadily increasing across many hops = general path congestion. Worth distinguishing when describing the issue to whoever you escalate to.
- **Reaching the destination successfully is itself diagnostic** — it rules out "below network layer" problems like a dead route or a firewalled destination, and (combined with other-offices-unaffected) rules out the destination's own infrastructure too.
- Full error/behavior reference for `ping`/`traceroute`/DNS tools lives in [9error-reference.ipynb](9error-reference.ipynb) if any output here looks unfamiliar later.